In [1]:
import cv2
import numpy as np
from deepface import DeepFace

MODEL_NAME = "ArcFace"
DETECTOR = "yunet"      # Nhẹ, chạy mượt trên webcam
THRESHOLD = 0.68        # Ngưỡng chuẩn của ArcFace (< 0.68 là khớp)

print("1. Đang tải model và trích xuất khuôn mặt từ ảnh mẫu...")
try:
    mau_res = DeepFace.represent(
        img_path="mau.jpg",
        model_name=MODEL_NAME,
        detector_backend=DETECTOR,
        enforce_detection=True
    )
    target_vec = np.asarray(mau_res[0]["embedding"])
    target_norm = np.linalg.norm(target_vec)
    print("-> Đã lưu đặc trưng khuôn mặt mẫu thành công!")
except Exception as e:
    print(f"Lỗi đọc ảnh mẫu (kiểm tra lại ánh sáng/độ rõ mặt): {e}")
    exit()

# Mở webcam (số 0 là camera mặc định của laptop)
cap = cv2.VideoCapture(0)
frame_count = 0
status_text = "Dang quet..."
box_color = (0, 255, 255) # Vàng
face_box = None

print("2. Đang mở camera... Nhấn phím 'Q' trên cửa sổ video để thoát.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Lật ngang khung hình cho giống soi gương
    frame = cv2.flip(frame, 1)

    # Chỉ chạy DeepFace mỗi 10 khung hình để webcam không bị giật lag
    if frame_count % 10 == 0:
        try:
            results = DeepFace.represent(
                img_path=frame,
                model_name=MODEL_NAME,
                detector_backend=DETECTOR,
                enforce_detection=True
            )
            # Lấy khuôn mặt đầu tiên tìm thấy
            face = results[0]
            face_box = face["facial_area"]
            
            # Tính khoảng cách Cosine với ảnh mẫu
            curr_vec = np.asarray(face["embedding"])
            distance = 1 - np.dot(target_vec, curr_vec) / (target_norm * np.linalg.norm(curr_vec))

            if distance < THRESHOLD:
                status_text = f"KHOP (Dist: {distance:.2f})"
                box_color = (0, 255, 0) # Xanh lá
            else:
                status_text = f"KHONG KHOP (Dist: {distance:.2f})"
                box_color = (0, 0, 255) # Đỏ
        except ValueError:
            # Không thấy mặt trong khung hình
            face_box = None
            status_text = "Khong thay mat"
            box_color = (0, 255, 255)

    # Vẽ khung chữ nhật và kết quả lên màn hình
    if face_box:
        x, y, w, h = face_box["x"], face_box["y"], face_box["w"], face_box["h"]
        cv2.rectangle(frame, (x, y), (x + w, y + h), box_color, 2)
        cv2.putText(frame, status_text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, box_color, 2)
    else:
        cv2.putText(frame, status_text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, box_color, 2)

    cv2.imshow("Test Nhan Dien Khuon Mat", frame)
    frame_count += 1

    # Nhấn Q để thoát
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


1. Đang tải model và trích xuất khuôn mặt từ ảnh mẫu...
26-11-19 22:35:24 - 🔗 arcface_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5 to C:\Users\Sang\.deepface\weights\arcface_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5
To: C:\Users\Sang\.deepface\weights\arcface_weights.h5
100%|██████████| 137M/137M [00:06<00:00, 21.2MB/s] 


26-11-19 22:35:33 - 🔗 face_detection_yunet_2023mar.onnx will be downloaded from https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx to C:\Users\Sang\.deepface\weights\face_detection_yunet_2023mar.onnx...


Downloading...
From: https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx
To: C:\Users\Sang\.deepface\weights\face_detection_yunet_2023mar.onnx
100%|██████████| 233k/233k [00:00<00:00, 1.08MB/s]


-> Đã lưu đặc trưng khuôn mặt mẫu thành công!
2. Đang mở camera... Nhấn phím 'Q' trên cửa sổ video để thoát.
